# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/protipa_exams_dataset` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [1]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [2]:
load_dotenv()

# API Config
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("LITELLM_HOST")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-21 20:06:06 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


In [3]:
# Ρυθμίσεις για να βρούμε τον κώδικα στο src
project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Προστέθηκε το {src_path} στο path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.")
except ImportError as e:
    logger.warning(f"⚠️ Δεν βρέθηκε η συνάρτηση/module. Έλεγξε τα ονόματα στο src. Error: {e}")

2026-01-21 20:06:09 - INFO - ✅ Προστέθηκε το c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\src στο path!
2026-01-21 20:06:09 - INFO - 🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.


## 2. Load and Prepare Dataset

In [4]:
#Script για εύκολη χρήση των νέων συναρτήσεων από το data_loader.py
from protipa_exams_dataset.data_loader import (
    load_protipa_dataset, 
    filter_dataset, 
    apply_matching_processing, 
    process_results_open
)

EVAL_MODE = 'open'

# Ρύθμιση φακέλου αποτελεσμάτων
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

logger.info("🚀 Ξεκινάει η διαδικασία σε Mode: {EVAL_MODE.upper()}...")

# --- Βήμα 1: Φόρτωση από Hugging Face ---
# Καλεί την έτοιμη συνάρτηση που κατεβάζει και ενώνει train + test
hf_dataset = load_protipa_dataset() 
logger.info(f"✅ Loaded raw dataset from HF. Size: {len(hf_dataset)}")

# --- Βήμα 2: Φιλτράρισμα ---
filtered_list = filter_dataset(hf_dataset, mode=EVAL_MODE)
logger.info(f"✅ Filtered items ({EVAL_MODE}): {len(filtered_list)}")

# --- Βήμα 3: Μετατροπή σε DataFrame ---
# Απαραίτητο για να δουλέψουν οι επόμενες συναρτήσεις (fix matching & clean paths)
df = pd.DataFrame(filtered_list)

# --- Βήμα 3b: ΔΙΟΡΘΩΣΗ ΤΩΝ ΕΙΚΟΝΩΝ ---
# Επειδή το HF μας δίνει PIL Objects (εικόνες) και εμείς θέλουμε Strings (ονόματα) για το JSON:
def extract_filename_from_object(val):
    # Αν είναι ήδη string (κείμενο), το κρατάμε
    if isinstance(val, str):
        return os.path.basename(val.replace('\\', '/'))
    # Αν είναι λίστα, το εφαρμόζουμε σε κάθε στοιχείο
    if isinstance(val, list):
        return [extract_filename_from_object(x) for x in val]
    # Αν είναι PIL Image (το PngImageFile που έβγαλε το error), παίρνουμε το filename του
    if hasattr(val, 'filename') and val.filename:
        return os.path.basename(val.filename.replace('\\', '/'))
    
    return None

logger.info("🖼️ Converting Image Objects to Filenames for JSON...")
if 'image' in df.columns:
    df['image'] = df['image'].apply(extract_filename_from_object)

# --- Βήμα 4: Διόρθωση Matching & Καθαρισμός ---
logger.info("🔄 Applying Matching Fix & Cleaning Paths...")
df = apply_matching_processing(df)  # Διορθώνει τα Matching

# --- Βήμα 5: Αποθήκευση ---
if EVAL_MODE == 'closed':
    json_filename = "full_dataset_for_eval_closed.json" 
else:
    json_filename = "full_dataset_for_eval_open.json" 

full_data_path = results_dir / json_filename

if not df.empty:
    df.to_json(full_data_path, orient="records", force_ascii=False, indent=4)
    logger.info(f"💾 Saved {EVAL_MODE} dataset to: {json_filename}")
    logger.info(f"✅ Questions ready for evaluation: {len(df)}")
else:
    logger.warning("⚠️ Το DataFrame είναι άδειο! Δεν αποθηκεύτηκε τίποτα.")

2026-01-21 20:06:15 - INFO - 🚀 Ξεκινάει η διαδικασία σε Mode: {EVAL_MODE.upper()}...
2026-01-21 20:06:15 - INFO - Loading dataset from Hugging Face: PennyK98/protipa_exams_dataset
2026-01-21 20:06:18 - INFO - ✅ Loaded raw dataset from HF. Size: 1646


🔍 Filtering for mode: OPEN...


2026-01-21 20:06:18 - INFO - ✅ Filtered items (open): 278
2026-01-21 20:06:18 - INFO - 🖼️ Converting Image Objects to Filenames for JSON...
2026-01-21 20:06:18 - INFO - 🔄 Applying Matching Fix & Cleaning Paths...
<unknown>:1: SyntaxWarning: invalid escape sequence '\G'
<unknown>:1: SyntaxWarning: invalid escape sequence '\G'
<unknown>:1: SyntaxWarning: invalid escape sequence '\e'
<unknown>:1: SyntaxWarning: invalid escape sequence '\G'
<unknown>:1: SyntaxWarning: invalid escape sequence '\o'
<unknown>:1: SyntaxWarning: invalid escape sequence '\o'
2026-01-21 20:06:18 - INFO - 💾 Saved open dataset to: full_dataset_for_eval_open.json
2026-01-21 20:06:18 - INFO - ✅ Questions ready for evaluation: 278


✅ Found 278 items for mode 'open'.


## 3. Define Evaluation Task Template

In [5]:
path_str = str(full_data_path)
logger.info(f"Setting up task with data file: {path_str}")

if EVAL_MODE == 'closed':
    task_config = {
        "task": "greek_protipa_exams",
        "dataset_path": "json",
        "num_fewshot": 0,      
        "dataset_kwargs": {
                "data_files": path_str
            },
        "test_split": "train",
        "output_type": "generate_until",
        "doc_to_text": (
            "{% if input %}{{input}}\n{% endif %}"
            "Ερώτηση: {{question}}\n"
            "Επιλογές:\n"
            "{% for choice in choices %}"
            "{{loop.index0}}. {{choice}}\n"
            "{% endfor %}\n"
            "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
            "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής (π.χ. 0, 1, 2, 3...).\n"
            "ΠΡΟΣΟΧΗ: Ο αριθμός '2' που χρησιμοποιείται στα παραδείγματα παρακάτω είναι ΤΥΧΑΙΟΣ και αφορά μόνο τη ΜΟΡΦΗ της απάντησης εδώ.\n"
            "Η σωστή απάντηση εξαρτάται αποκλειστικά από την ερώτηση και μπορεί να είναι οποιοσδήποτε αριθμός.\n"
            "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
            "Παραδείγματα Μορφής:\n"
            "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
            "❌ ΛΑΘΟΣ: \"(2)\"\n"
            "✅ ΣΩΣΤΟ: 2 (ή 0 ή 1 ή 3... ανάλογα με τη σωστή επιλογή)\n\n"
            "Απάντηση: "
        ),
        "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
        "generation_kwargs": {
            "until": ["\n"],
            "max_gen_toks": 50,
            "do_sample": False,
            "temperature": 0.0 
        },
        "filter_list": [
            {
                "name": "strict-match",
                "filter": [
                    {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                    {"function": "take_first"}
                ]
            }
        ],
        "metric_list": [
            {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
        ]
    }

else: # EVAL_MODE == 'open'
    task_config = {
        "task": "greek_protipa_exams_open",
        "dataset_path": "json",
        "num_fewshot": 0,      
        "dataset_kwargs": {
            "data_files": path_str
        },
        "test_split": "train",
        "output_type": "generate_until",
        "doc_to_text": (
            "Δίνεται η παρακάτω ερώτηση από σχολικές εξετάσεις.\n"
            "Αν είναι ερώτηση ανάπτυξης, δώσε μια ολοκληρωμένη και τεκμηριωμένη απάντηση.\n"
            "Αν είναι ερώτηση συμπλήρωσης κενών, γράψε τη σωστή λέξη ή τη σωστή φράση που λείπει.\n\n"
            "{% if input %}Πλαίσιο/Κείμενο: {{input}}\n{% endif %}"
            "Ερώτηση: {{question}}\n\n"
            "Απάντηση:"
        ),
        "doc_to_target": "{{ answer }}",
        "process_results": process_results_open,
        "generation_kwargs": {
            "until": ["Ερώτηση:", "---"], # Σταματάει αν πάει να ξεκινήσει νέα ερώτηση
            "max_gen_toks": 512,          # Δίνουμε χώρο για ανάπτυξη
            "do_sample": False,
            "temperature": 0.0 
        },
        "metric_list": [
            {
                "metric": "bleu", 
                "aggregation": "mean", 
                "higher_is_better": True
            },
            {
                "metric": "chrf", 
                "aggregation": "mean", 
                "higher_is_better": True
            }
        ]
    }

# --- Αποθήκευση και Φόρτωση ---
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)

yaml_filename = f"greek_protipa_{EVAL_MODE}.yaml"

with open(task_dir / yaml_filename, "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_name = f"greek_protipa_exams_{EVAL_MODE}"
task_dict = {task_name: custom_task}

logger.info(f"Evaluation task '{task_name}' defined successfully.")

2026-01-21 20:07:25 - INFO - Setting up task with data file: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\full_dataset_for_eval_open.json


Generating train split: 0 examples [00:00, ? examples/s]

2026-01-21 20:07:26 - INFO - Evaluation task 'greek_protipa_exams_open' defined successfully.


## 4. Run Evaluation

In [6]:
from protipa_exams_dataset.evaluation import run_evaluation

comparison_results = {}
all_samples = {}

EVAL_LIMIT = 20  # Adjust this to run more/less samples

for model_name in models_to_test:
    
    results = run_evaluation(
        model_name=model_name,
        api_base=api_base,
        task_dict=task_dict,
        eval_limit=EVAL_LIMIT
    )

    if results is None:
        continue

    try:
        scores = results['results'][task_name]
        comparison_results[model_name] = scores
        
        if 'samples' in results and task_name in results['samples']:
            samples = results['samples'][task_name]
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    if EVAL_MODE == 'open':
                        ground_truth = str(doc.get('answer', 'N/A'))
                    else:
                        ground_truth = str(doc.get('answer_index', 'N/A'))
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Exercise Type": doc.get('exercise_type', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('school_level', 'N/A'),
                        "Ground Truth": ground_truth,
                        "Choices": " | ".join(doc.get('choices', [])),
                        "Multimodality": doc.get('multimodality', 'no'),
                        "Image Path": str(doc.get('image', '')),
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        if EVAL_MODE == 'open':
            chrf = scores.get('chrf', scores.get('score', 0))
            bleu = scores.get('bleu', 0)
            #rouge_l = scores.get('rougeL', scores.get('rougeL,none', 0))
            
            #if rouge_l == 0:
                #rouge_l = scores.get('rouge', 0)
            
            logger.info(f"✅ Success! {model_name} Results:")
            logger.info(f"   🔹 ChrF: {chrf:.4f}")
            logger.info(f"   🔹 BLEU: {bleu:.4f}")
            #logger.info(f"   🔹 ROUGE-L: {rouge_l:.4f}")
            
        else:
            acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
            logger.info(f"✅ Success! {model_name} Accuracy: {acc:.2%}")

        logger.info("⏳ Waiting 2 seconds before next model...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"❌ Error processing results for {model_name}: {e}")
        logger.error(traceback.format_exc())

2026-01-21 20:07:35 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-21 20:07:35 - INFO - Using max length 2048 - 1
2026-01-21 20:07:35 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-21 20:07:35 - INFO - Using tokenizer None
2026-01-21 20:07:35 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-21 20:07:35 - INFO - Building contexts for greek_protipa_exams_open on rank 0...
100%|██████████| 20/20 [00:00<00:00, 1609.17it/s]
2026-01-21 20:07:35 - INFO - Running generate_until requests
2026-01-21 20:07:35 - INFO - Tokenized requests are disabled. Context + generation length is not checked.
Requesting API:   0%|          | 0/20 [00:00<?, ?it/s]2026-01-21 20:12:01 - ERROR - Error evaluating gemma3-27b-it: HTTPConnectionPool(host='ec2-3-18-183-221.us-east-2.compute.amazonaws.com', port=4000): Max retries exceeded with url: /

KeyboardInterrupt: 

## 5. Results Table

In [7]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():
        row = {
            "ID": idx,
            "Subject": data.get("Subject"),
            "Exercise Type": data.get("Exercise Type", "N/A"),
            "Multimodality": data.get("Multimodality", "no"), 
            "Image Path": data.get("Image Path", ""),         
            "Year": data.get("Year"),
            "Level": data.get("Level", "N/A"),
            "Question": data.get("Question", ""),
            "Choices": data.get("Choices", ""),
            "Ground Truth": data.get("Ground Truth", "")
        }
        
        for m in models_to_test:
            # Αποθηκεύουμε την απάντηση του κάθε μοντέλου
            model_col_name = m.split("/")[-1]
            pred = data["Model Predictions"].get(m, "N/A")
            row[f"{model_col_name}_pred"] = pred
            # row[f"{m}_raw"] = data["Raw Responses"].get(m, "N/A") 
            
        table_data.append(row)
    
    df_results = pd.DataFrame(table_data)
    
    model_names_str = "_".join([m.split("/")[-1] for m in models_to_test])
    filename = f"eval_results_{model_names_str}_{EVAL_MODE}.csv"
    
    results_file = results_dir / filename

    # Αποθήκευση
    df_results.to_csv(results_file, index=False, encoding='utf-8-sig')
    logger.info(f"💾 Table saved successfully to: {results_file}")
    
    display(df_results.head())
else:
    logger.warning("⚠️ No samples collected to save!")

2026-01-21 19:31:04 - INFO - 💾 Table saved successfully to: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\eval_results_gemma3-27b-it_krikri-dpo-context_open.csv


,ID,Subject,Exercise Type,Multimodality,Image Path,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,modern greek,open,no,[],2016,lyceum,Σύμφωνα με το κείμενο 1 να προσδιορίσετε 2 λόγ...,,"[""Α. Τα θεωρούν κάτι σαν μαυσωλεία αποθήκευσης...","Σύμφωνα με το κείμενο 1, δύο λόγοι για τους οπ...","Σύμφωνα με το παρεχόμενο κείμενο, δύο σημαντικ..."
1,1,modern greek,fill-in-the-gaps,no,[],2018,gymnasium,Συμπληρώνω το κενό γράφοντας τον σωστό τύπο: «...,,ανάρτησαν,Απάντηση: **Ανέβασαν**\n,Η σωστή απάντηση είναι: **ανέβασαν**\n\nΕξήγησ...
2,2,mathematics,open,no,[],2014,lyceum,Υπάρχει πραγματικός αριθμός που να ικανοποιεί ...,,Η σχέση $x^{2}+25=0$ γράφεται $x^{2}=-25$.\nΔε...,"Όχι, δεν υπάρχει πραγματικός αριθμός που να ικ...",Η ερώτηση αφορά την ύπαρξη λύσεων σε μια δευτε...
3,3,modern greek,open,no,[],2013,lyceum,"Δεν γνώριζαν κανένα. Μαζεύονταν στο προαύλιο, ...",,"Καθώς δεν γνώριζαν κανένα, μαζεύονταν στο προα...",Η ερώτηση είναι ερώτηση ανάπτυξης και απαιτεί ...,Για να συνδέσει ο συγγραφέας αποτελεσματικά τι...
4,4,modern greek,fill-in-the-gaps,no,[],2018,gymnasium,"Συμπληρώστε το κενό, βάζοντας τη λέξη που δίνε...",,καταβάλει,καταβάλει\n,"Η σωστή συμπλήρωση του κενού είναι:\n\n""καταβά..."


## 6. Performance Summary

In [ ]:
if comparison_results:
    summary_data = []
    
    for model, metrics in comparison_results.items():
        row = {"Model": model}
        
        if EVAL_MODE == 'open':
            chrf = metrics.get('chrf', metrics.get('score', 0))
            bleu = metrics.get('bleu', 0)
            
            row['ChrF'] = chrf
            row['BLEU'] = bleu
        else:
            acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
            row['Accuracy'] = acc
        
        summary_data.append(row)
    
    df_summary = pd.DataFrame(summary_data)
    
    # Αποθήκευση με δυναμικό όνομα για να μην σβήνουμε τα προηγούμενα!
    #summary_filename = f"evaluation_summary_scores_{EVAL_MODE}.csv"
    #summary_file = results_dir / summary_filename
    
    #df_summary.to_csv(summary_file, index=False)
    #logger.info(f"📊 Summary saved to: {summary_file}")

    print(f"\n=== ΤΕΛΙΚΗ ΒΑΘΜΟΛΟΓΙΑ ({EVAL_MODE.upper()}) ===")
    
    # Formatting για ωραία εκτύπωση
    df_display = df_summary.copy()
    if 'Accuracy' in df_display.columns:
        df_display['Accuracy'] = df_display['Accuracy'].apply(lambda x: f"{x:.2%}")
    if 'ChrF' in df_display.columns:
        df_display['ChrF'] = df_display['ChrF'].apply(lambda x: f"{x:.4f}")
    if 'BLEU' in df_display.columns:
        df_display['BLEU'] = df_display['BLEU'].apply(lambda x: f"{x:.4f}")
        
    display(df_display)

else:
    logger.warning("⚠️ No comparison results found to summarize.")

,Model,Accuracy
0,gemma3-27b-it,0.614035
1,krikri-dpo-context,0.389620


In [ ]:
# Φορτώνουμε το CSV που μόλις φτιάξαμε
if results_file.exists():
    df = pd.read_csv(results_file)
    print(f"📊 Φορτώθηκαν {len(df)} ερωτήσεις για ανάλυση.\n")

    if EVAL_MODE == 'closed':
        print("🔒 Mode is CLOSED: Calculating binary Accuracy (Correct/Incorrect)...")
        
        def normalize_val(val):
            """Καθαρίζει την τιμή για να γίνει σωστή σύγκριση."""
            s = str(val).strip()
            # Αν κατά λάθος έγινε 1.0 (float string), το κάνουμε 1
            if s.endswith(".0"):
                s = s[:-2]
            return s

        # Υπολογισμός Σωστού/Λάθους δυναμικά
        for model in models_to_test:
            col_name = f"{model.split('/')[-1]}_pred" 
            
            if col_name in df.columns:
                gt_clean = df["Ground Truth"].apply(normalize_val)
                pred_clean = df[col_name].apply(normalize_val)
                
                df[f"{model.split('/')[-1]}_correct"] = (gt_clean == pred_clean).astype(int)
                print(f"   ✅ Calculated accuracy column for: {model}")
                
        # Αποθήκευση μόνο αν κάναμε αλλαγές
        #df.to_csv(results_file, index=False, encoding='utf-8-sig')
        #print(f"💾 Updated CSV saved to: {results_file}")

    else:
        print("🔓 Mode is OPEN: Skipping binary correctness calculation.")
        print("   ℹ️ Στα Open tasks δεν υπάρχει απόλυτο 0 ή 1.")

else:
    print("⚠️ Δεν βρέθηκε το αρχείο αποτελεσμάτων.")

📊 Φορτώθηκαν 1368 ερωτήσεις για ανάλυση.

✅ Calculated accuracy column for: gemma3-27b-it
✅ Calculated accuracy column for: krikri-dpo-context


Ανάλυση RQ1: Ακρίβεια ανά μάθημα (κλειστού τύπου)

In [13]:
# Ανάλυση RQ1: Ακρίβεια ανά Είδος Άσκησης (Μάθημα)
print("\n" + "-" * 60)
print("🏆 RQ1: PERFORMANCE PER SUBJECT")
print("-" * 60)

model_cols = [c for c in df.columns if c.endswith('_correct')]

if not model_cols:
    logger.warning("⚠️ Δεν βρέθηκαν στήλες αποτελεσμάτων (_correct).")
else:
    #Υπολογίζουμε τη μέση τιμή (Accuracy) για κάθε μάθημα
    rq1_acc = df.groupby("Subject")[model_cols].mean()
    
    #Υπολογίζουμε το πλήθος των ερωτήσεων ανά μάθημα
    rq1_count = df.groupby("Subject")[model_cols[0]].count()
    
    rq1_final = pd.DataFrame()
    
    rq1_final['Total Questions'] = rq1_count
    
    for col in model_cols:
        clean_name = col.replace("_correct", "")
        rq1_final[clean_name] = (rq1_acc[col] * 100).round(1).astype(str) + '%'

    display(rq1_final)
    
    #rq1_csv_path = results_dir / "RQ1_performance_per_subject.csv"
    #rq1_final.to_csv(rq1_csv_path)
    #logger.info(f"💾 RQ1 Table saved to: {rq1_csv_path}")

logger.info("Evaluation analysis for RQ1 completed.")


------------------------------------------------------------
🏆 RQ1: PERFORMANCE PER SUBJECT
------------------------------------------------------------


,Total Questions,gemma3-27b-it,krikri-dpo-context
Subject,,,
mathematics,555,44.3%,26.3%
modern greek,723,71.6%,45.2%
religious studies,90,77.8%,64.4%


2026-01-20 11:30:06 - INFO - Evaluation analysis for RQ1 completed.


Ανάλυση RQ2: Απόδοση ανάλογα με το αν υπάρχει ή όχι εικόνα (κλειστού τύπου)

In [14]:
# Ανάλυση RQ2: Multimodality (Εικόνα vs Κείμενο)
print("\n" + "-" * 60)
print("🏆 RQ2: IMPACT OF MULTIMODALITY")
print("-" * 60)

# Βρίσκουμε τις στήλες των μοντέλων
model_cols = [c for c in df.columns if c.endswith('_correct')]

# Ομαδοποίηση
rq2_stats = df.groupby("Multimodality")[model_cols].mean()
rq2_count = df.groupby("Multimodality")[model_cols[0]].count()

# Μορφοποίηση
rq2_final = pd.DataFrame()
rq2_final["Total Questions"] = rq2_count

for col in model_cols:
    clean_name = col.replace("_correct", "")
    rq2_final[clean_name] = (rq2_stats[col] * 100).round(1).astype(str) + '%'

display(rq2_final)

# Αποθήκευση
#rq2_final.to_csv(results_dir / "RQ2_multimodality_impact.csv")


------------------------------------------------------------
🏆 RQ2: IMPACT OF MULTIMODALITY
------------------------------------------------------------


,Total Questions,gemma3-27b-it,krikri-dpo-context
Multimodality,,,
no,1176,63.5%,40.6%
yes,192,45.3%,28.1%


Ανάλυση RQ3: Ακρίβεια ανά είδος άσκησης (κλειστού τύπου)

In [15]:
# Ανάλυση RQ3: Ακρίβεια ανά Είδος Άσκησης
print("\n" + "-" * 60)
print("🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE")
print("-" * 60)

model_cols = [c for c in df.columns if c.endswith('_correct')]

if not model_cols:
    logger.warning(f"⚠️ Δεν βρέθηκαν στήλες αποτελεσμάτων (_correct).")

else:
    rq3_acc = df.groupby("Exercise Type")[model_cols].mean()
    rq3_count = df.groupby("Exercise Type")[model_cols[0]].count()
    
    rq3_final = pd.DataFrame()
    rq3_final["Total Exercises"] = rq3_count
    
    for col in model_cols:
        clean_name = col.replace("_correct", "")
        
        rq3_final[clean_name] = (rq3_acc[col] * 100).round(1).astype(str) + '%'
    
    display(rq3_final)
    
    #rq3_csv_path = results_dir / "RQ3_performance_per_exercise_type.csv"
    #rq3_final.to_csv(rq3_csv_path)
    #logger.info(f"💾 RQ3 Table saved to: {rq3_csv_path}")

logger.info("Evaluation analysis for RQ3 completed.")


------------------------------------------------------------
🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE
------------------------------------------------------------


,Total Exercises,gemma3-27b-it,krikri-dpo-context
Exercise Type,,,
fill-in-the-gaps,9,88.9%,77.8%
matching,4,0.0%,0.0%
multiple choice,1268,59.3%,37.1%
true/false,87,85.1%,60.9%


2026-01-20 11:31:05 - INFO - Evaluation analysis for RQ3 completed.


Διαθέσιμα μοντέλα

In [ ]:
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            # Αν το ID περιέχει κάποια από τις λέξεις κλειδιά
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "mistral-large-24.02",
    "gpt-oss-120b",
    "mistral-small-24.02",
    "mistral-7b-instruct-v0.2",
    "llama-3.2-1b",
    "gemma3-27b-it",
    "llama-3.1-8b",
    "llama-3.3-70b",
    "gemma3-27b-it-long",
    "llama-3.2-3b",
    "gpt-4o-mini",
    "gpt-4o",
    "llama-3.1-70b",
    "gpt-oss-20b"
]


**ΠΑΡΑΤΗΡΗΣΕΙΣ**

○ Τα υπόλοιπα μοντέλα (πχ llama-3.1-8b, mistral-7b-instruct-v0.2) σκάνε με Bedrock error όταν τα τρέχω

***ΟΛΑ ΤΑ ΜΑΘΗΜΑΤΑ, all closed-type questions*** (RQ1: Comparative Performance Across Subjects)

○ **Στο σύνολο των δεδομένων**, το gemma έχει accuracy 61% ενώ το krikri είχε 39%.

○ Στα επιμέρους μαθήματα παρατηρούνται τα εξής:

| Subject | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| ΓΛΩΣΣΑ | 723 | 71.6% | 45.2% |
| ΘΡΗΣΚΕΥΤΙΚΑ | 90 | 77.8% | 64.4% |
| ΜΑΘΗΜΑΤΙΚΑ | 555 | 44.3% | 26.3% |

***Multimodality VS Κείμενο*** (RQ2)

○ **Στο σύνολο των δεδομένων** παρατηρούνται τα εξής:

| Multimodality | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| no | 1176 | 63.5% | 40.6% |
| yes | 192 | 45.3% | 28.1% |

○ Για το gemma έχουμε πτώση 18.2% με την παρουσία εικόνας, ενώ για το krikri έχουμε πτώση μόλις 12.5%

***Αll types of closed questions (multiple-choice, fill-in-the-gaps, true/false, matching)*** (RQ3: Performance Across Exercise Types)

○ Ο τωρινός κώδικας (task_config) είναι φτιαγμένος να ψάχνει για ένα index, δηλαδή έναν αριθμό (π.χ. 1, 2, 3). Αυτό δουλεύει στο Multiple Choice, στο True/False και στο Fill-in-the-gaps γιατί εκεί η σωστή απάντηση είναι "Επιλογή 1" ή "Επιλογή 2". Στο Matching, η απάντηση δεν είναι ένας αριθμός. Είναι μια λίστα με ζεύγη (["1-γ", "2-α", "3-ε", "4-β", "5-δ"]). Στο dataset, το πεδίο answer_index είναι κενό (None) γιατί δεν υπάρχει "μία σωστή επιλογή", αλλά ένας συνδυασμός. Καλό θα ήταν να εξαιρέσουμε το Matching από αυτό το πείραμα (Closed Types), καθώς χρειάζεται άλλο κώδικα αξιολόγησης και πρόκειται για ελάχιστα παραδείγματα.

○ **Στο σύνολο των δεδομένων**, στα διαφορετικά είδη ερωτήσεων κλειστού τύπου παρατηρούνται τα εξής:

| Exercise Type    | Total Exercises | gemma | krikri |
| :--- | :---: | :---: | :---: |
| Fill-in-the-gaps | 9               | 88.9% | 77.8%  |
| Matching         | 4               | 0.0%  | 0.0%   |
| Multiple Choice  | 1268            | 59.3% | 37.1%  |
| True/False       | 87              | 85.1% | 60.9%  |

Πρόβλημα με matching ερωτήσεις

In [ ]:
# Έλεγχος: Δείξε μου μια Matching ερώτηση όπως είναι ΤΩΡΑ στο df
matching_check = df[df['Exercise Type'] == 'Matching'].head(1)

if not matching_check.empty:
    print("✅ Βρέθηκε Matching ερώτηση.")
    print("\n--- Choices (Επιλογές που βλέπει το μοντέλο) ---")
    # Τυπώνουμε την πρώτη εγγραφή της στήλης Choices
    print(matching_check['Choices'].iloc[0]) 
    
    print("\n--- Ground Truth (Σωστή απάντηση) ---")
    print(matching_check['Ground Truth'].iloc[0])
    
    print("\n--- Τι απάντησαν τα μοντέλα ---")
    for model in models_to_test:
        col = f"{model}_pred"
        if col in matching_check.columns:
            print(f"{model}: {matching_check[col].iloc[0]}")
else:
    print("Δεν βρέθηκαν Matching ερωτήσεις στο df")

✅ Βρέθηκε Matching ερώτηση.

--- Choices (Επιλογές που βλέπει το μοντέλο) ---
Α. θέατρο | Β. χορός | Γ. πανηγύρια | α. συναναστροφή | β. λατρεία | γ. διασκέδαση | δ. εύθυμη διάθεση | ε. καλή σωματική κατάσταση | στ. καλλιέργεια πνεύματος

--- Ground Truth (Σωστή απάντηση) ---
nan

--- Τι απάντησαν τα μοντέλα ---
gemma3-27b-it: 0
krikri-dpo-context: 0
